# Solution 4b — Robustness of the fine-tuning result

Solution 4 gave the first significant mitigation (+0.043 Darija R@5, CI [0.005, 0.080], no MSA regression). Three checks before it can go in a paper:

**1. Subset.** In the single run the gain came from the *pilot* subset (0.864 → 0.941) while Wikipedia went slightly **down** (0.942 → 0.928). Pilot is the confounded one. If the effect only lives there, the claim collapses.

**2. Seed.** One split is one sample. Five seeds show whether it's stable.

**3. Ablation.** Which objective does the work — retrieval supervision (darija→passage) or representation alignment (darija→MSA)?

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.**

⏱️ **Longest notebook in the project** — 15 fine-tuning runs, roughly 1.5–3 hours on a T4. Checkpointed after every run, so a dropped session resumes where it stopped.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests accelerate

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "repo_raw": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    "seeds": [42, 7, 123, 2024, 777],
    "ablations": ["both", "passage_only", "paraphrase_only"],

    "test_frac": 0.35,
    "epochs": 3,
    "batch_size": 16,
    "lr": 2e-5,
    "warmup_frac": 0.1,

    "k_values": (1, 5),
    "max_k": 10,
    "bootstrap_n": 1000,
    "ci": 95,
    "checkpoint": "robustness_checkpoint.json",
}

### Load data

In [ ]:
import json, requests, random, re, os, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)
pilot_qa = json.loads(requests.get(f"{CONFIG['repo_raw']}/data/qa_pairs.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

all_qa = [q for q in (wiki_qa + pilot_qa) if q["source_chunk_id"] in known]

def subset_of(q):
    return "wikipedia" if q["source_chunk_id"].startswith("wiki_") else "pilot"

print(f"Corpus {len(corpus)} | QA {len(all_qa)} "
      f"(wikipedia {sum(1 for q in all_qa if subset_of(q)=='wikipedia')}, "
      f"pilot {sum(1 for q in all_qa if subset_of(q)=='pilot')})")

### BM25 + Arabic normalization

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
# BM25 scores are query-dependent but corpus-fixed, so cache per query string:
# the same queries are evaluated 30 times across runs.
_bm25_cache = {}
def bm25_scores(q):
    if q not in _bm25_cache:
        _bm25_cache[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm25_cache[q]

print("BM25 ready.")

### Split, training pairs, retrieval, evaluation

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch

def make_split(seed):
    """Split by SOURCE PASSAGE so no passage appears in both train and test."""
    rnd = random.Random(seed)
    passages = sorted({q["source_chunk_id"] for q in all_qa})
    rnd.shuffle(passages)
    n_test = int(len(passages) * CONFIG["test_frac"])
    test_p = set(passages[:n_test])
    train = [q for q in all_qa if q["source_chunk_id"] not in test_p]
    test = [q for q in all_qa if q["source_chunk_id"] in test_p]
    assert not ({q["source_chunk_id"] for q in train} & test_p), "passage leakage"
    return train, test

def make_examples(train_qa, ablation, seed):
    ex = []
    for q in train_qa:
        if ablation in ("both", "passage_only"):
            ex.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                          f'passage: {corpus_map[q["source_chunk_id"]]}']))
        if ablation in ("both", "paraphrase_only"):
            ex.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                          f'query: {q["msa_query"]}']))
    random.Random(seed).shuffle(ex)
    return ex

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def build_index(model):
    emb = model.encode([f"passage: {t}" for t in corpus_texts],
                       normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    return np.asarray(emb, "float32")

def evaluate(model, emb, items, field):
    a = CONFIG["alpha"]
    hits = {k: [] for k in CONFIG["k_values"]}
    rr = []
    qs = [it[field] for it in items]
    qemb = model.encode([f"query: {q}" for q in qs],
                        normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    for i, it in enumerate(items):
        s = a * minmax(emb @ qemb[i]) + (1 - a) * minmax(bm25_scores(qs[i]))
        got = [corpus_ids[j] for j in np.argsort(-s)[:CONFIG["max_k"]]]
        gold = it["source_chunk_id"]
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if gold in got[:k] else 0.0)
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()}, "MRR": np.array(rr)}

def finetune(train_examples):
    model = SentenceTransformer(CONFIG["base_encoder"])
    loader = DataLoader(train_examples, shuffle=True, batch_size=CONFIG["batch_size"])
    loss = losses.MultipleNegativesRankingLoss(model)
    steps = len(loader) * CONFIG["epochs"]
    model.fit(train_objectives=[(loader, loss)], epochs=CONFIG["epochs"],
              warmup_steps=int(steps * CONFIG["warmup_frac"]),
              optimizer_params={"lr": CONFIG["lr"]}, show_progress_bar=False)
    return model

### Run the grid (checkpointed)

In [ ]:
records = []
if os.path.exists(CONFIG["checkpoint"]):
    records = json.load(open(CONFIG["checkpoint"], encoding="utf-8"))
    print(f"Resuming with {len(records)} records already done.")

done = {(r["seed"], r["ablation"]) for r in records}
total = len(CONFIG["seeds"]) * len(CONFIG["ablations"])

for seed in CONFIG["seeds"]:
    train_qa, test_qa = make_split(seed)
    idx_by_subset = {
        name: np.array([i for i, q in enumerate(test_qa) if subset_of(q) == name])
        for name in ["wikipedia", "pilot"]
    }

    # Baseline is identical across ablations for a given seed, so compute once.
    base = SentenceTransformer(CONFIG["base_encoder"])
    base_emb = build_index(base)
    before = {f: evaluate(base, base_emb, test_qa, f) for f in ["msa_query", "darija_query"]}
    del base, base_emb
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

    for ablation in CONFIG["ablations"]:
        if (seed, ablation) in done:
            continue
        print(f"\n[{len(records)+1}/{total}] seed={seed} ablation={ablation} "
              f"(train {len(train_qa)} / test {len(test_qa)})")

        model = finetune(make_examples(train_qa, ablation, seed))
        emb = build_index(model)
        after = {f: evaluate(model, emb, test_qa, f) for f in ["msa_query", "darija_query"]}

        rec = {"seed": seed, "ablation": ablation,
               "n_train": len(train_qa), "n_test": len(test_qa)}
        for metric in ["R@1", "R@5", "MRR"]:
            rec[f"all_darija_before_{metric}"] = float(before["darija_query"][metric].mean())
            rec[f"all_darija_after_{metric}"]  = float(after["darija_query"][metric].mean())
            rec[f"all_msa_before_{metric}"]    = float(before["msa_query"][metric].mean())
            rec[f"all_msa_after_{metric}"]     = float(after["msa_query"][metric].mean())
            for name, idx in idx_by_subset.items():
                if len(idx) == 0:
                    continue
                rec[f"{name}_n"] = int(len(idx))
                rec[f"{name}_darija_before_{metric}"] = float(before["darija_query"][metric][idx].mean())
                rec[f"{name}_darija_after_{metric}"]  = float(after["darija_query"][metric][idx].mean())
                rec[f"{name}_msa_after_{metric}"]     = float(after["msa_query"][metric][idx].mean())
        records.append(rec)
        json.dump(records, open(CONFIG["checkpoint"], "w", encoding="utf-8"), indent=2)

        del model, emb
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

df = pd.DataFrame(records)
df.to_csv("robustness_raw.csv", index=False)
print(f"\nComplete: {len(df)} runs -> robustness_raw.csv")

### (1) Does the effect hold across seeds?

In [ ]:
print("=" * 84)
print("SEED STABILITY — ablation 'both', Darija gain per seed")
print("=" * 84)
b = df[df.ablation == "both"]
for metric in ["R@1", "R@5", "MRR"]:
    gains = b[f"all_darija_after_{metric}"] - b[f"all_darija_before_{metric}"]
    pos = (gains > 0).sum()
    print(f"  {metric:<5} mean {gains.mean():+.4f}  sd {gains.std():.4f}  "
          f"range [{gains.min():+.4f}, {gains.max():+.4f}]  positive in {pos}/{len(gains)} seeds")

print("\nAn effect positive in most seeds with sd well below the mean is stable.")
print("An effect that flips sign across seeds is not a result.")

### (2) THE KEY TEST: does it hold on the unconfounded subset?

In [ ]:
print("\n" + "=" * 84)
print("PER-SUBSET — is the gain driven only by the confounded pilot subset?")
print("=" * 84)
rows = []
for name in ["wikipedia", "pilot"]:
    for metric in ["R@1", "R@5", "MRR"]:
        g = b[f"{name}_darija_after_{metric}"] - b[f"{name}_darija_before_{metric}"]
        rows.append({"subset": name, "metric": metric, "mean_gain": g.mean(),
                     "sd": g.std(), "positive_seeds": f"{(g > 0).sum()}/{len(g)}"})
per_subset = pd.DataFrame(rows)
print(per_subset.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
per_subset.to_csv("robustness_per_subset.csv", index=False)

print("\nIf the wikipedia rows are positive in most seeds, the effect is real.")
print("If only pilot is positive, the headline claim must be restricted to it")
print("and the authorship confound stated as the likely cause.")

### (3) Which training objective does the work?

In [ ]:
print("\n" + "=" * 84)
print("ABLATION — contribution of each training objective")
print("=" * 84)
abl = []
for a in CONFIG["ablations"]:
    s = df[df.ablation == a]
    for metric in ["R@1", "R@5"]:
        g = s[f"all_darija_after_{metric}"] - s[f"all_darija_before_{metric}"]
        msa = s[f"all_msa_after_{metric}"] - s[f"all_msa_before_{metric}"]
        abl.append({"ablation": a, "metric": metric,
                    "darija_gain": g.mean(), "darija_sd": g.std(),
                    "msa_change": msa.mean()})
abl_df = pd.DataFrame(abl)
print(abl_df.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
abl_df.to_csv("robustness_ablation.csv", index=False)

print("\npassage_only   = darija -> gold passage (retrieval supervision)")
print("paraphrase_only = darija -> msa question (representation alignment)")
print("A large msa_change (negative) means the model traded MSA for Darija.")

### Headline numbers for the paper

In [ ]:
print("\n" + "=" * 84)
print("PAPER NUMBERS — mean over seeds, ablation 'both'")
print("=" * 84)
for metric in ["R@1", "R@5", "MRR"]:
    db, da = b[f"all_darija_before_{metric}"].mean(), b[f"all_darija_after_{metric}"].mean()
    mb, ma = b[f"all_msa_before_{metric}"].mean(), b[f"all_msa_after_{metric}"].mean()
    print(f"  {metric:<5} Darija {db:.3f} -> {da:.3f} ({da-db:+.3f})   "
          f"gap {mb-db:.3f} -> {ma-da:.3f}   MSA change {ma-mb:+.3f}")

from google.colab import files
files.download("robustness_raw.csv")
files.download("robustness_per_subset.csv")
files.download("robustness_ablation.csv")